# Module 4 — Model Context Protocol: Topics & Practice

## Learning outcomes
1. Explain MCP in one paragraph and name its three primitives.
2. Build an MCP server that exposes tools, resources, and prompts.
3. Connect that server to a host (Claude Desktop / Code) and use it from a chat.
4. Design tool schemas that don't blow up at the edges (auth, idempotency, errors).

## 1. What is MCP?

MCP is a JSON-RPC-based **protocol** standardising how an AI host (Claude, ChatGPT desktop, an IDE) talks to external **servers** that provide:

- **Tools** — callable functions the model can invoke (`create_ticket`, `query_db`).
- **Resources** — readable content (files, DB rows, API responses) the host can pull as context.
- **Prompts** — reusable prompt templates the user can pick from a menu.

Think of MCP as **USB-C for AI tooling**: any compliant server plugs into any compliant host.

> **Exercise 4.1** — In one sentence each, contrast MCP with: (a) OpenAI function calling, (b) LangChain tools, (c) plain REST. Why does standardising matter?

## 2. Server anatomy

```
server
├── transport          stdio | http+sse | websocket
├── capabilities       which primitives this server supports
├── tools              name, description, JSON-Schema input
├── resources          uri, name, mimeType
└── prompts            name, description, arguments
```

The host calls `initialize`, asks for `tools/list`, then `tools/call`. Your server is just a JSON-RPC handler.

> **Exercise 4.2** — Sketch a tool schema for `create_calendar_event(title, start, end, attendees)`. What goes in `description`? What edge inputs would you reject?

## 3. Tool design — five rules that prevent pain

1. **Idempotent where possible.** A retried `create_user(email)` shouldnt make two users.
2. **Errors are typed.** Return `{error: {code, message}}`, not free-text.
3. **Auth scoping is per-call.** The model gets the *users* token, not yours.
4. **Cap blast radius.** No `delete_all`. Add `dry_run`, confirmations, rate limits.
5. **Logs are mandatory.** Every call: who, what, when, args, outcome.

> **Exercise 4.3** — Pick one tool from your imagined LMS server (`enroll_user`, `issue_certificate`, `delete_account`). Apply all five rules. Write the schema and the threat model.

## 4. Client / host integration

- **Claude Desktop**: add to `~/Library/Application Support/Claude/claude_desktop_config.json`.
- **Claude Code**: `claude mcp add <name> <command>` or in `.mcp.json`.
- **Custom host**: spawn the server as a subprocess (stdio) or open an HTTP+SSE connection.

The host handles permission prompts; your server provides metadata to make those prompts informative.

> **Exercise 4.4** — Read the `claude_desktop_config.json` schema. Whats the minimum entry to register a Python-based stdio MCP server?